# WaveSeekerNet Demonstration with Real IAV Data

This notebook demonstrates how to use the `waveseekernet` package for predicting Influenza A virus subtypes or host sources. 

We will:
1. Load data.
2. Initialize the `WaveSeekerClassifier`.
3. Perform a short training run.
4. Evaluate the model performance.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")

In [2]:
import numpy as np
import random
import torch
from WaveSeekerNet import WaveSeekerClassifier, fasta_to_one_hot, fasta_to_fcgr, get_rare_sequence, resampling, protein_fasta_to_one_hot
from sklearn.metrics import classification_report, balanced_accuracy_score
import shap
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import balanced_accuracy_score as ba_score
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, matthews_corrcoef

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.12.1+cu130
CUDA available: True


In [4]:
def set_seed(random_seed):
    print ("Set Global Seed\n")
    torch.manual_seed(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)

In [5]:
def get_score(y_true, y_pred):
    ba = ba_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    p_score = precision_score(y_true, y_pred, average="macro")
    r_score = recall_score(y_true, y_pred, average="macro")
    mcc = matthews_corrcoef(y_true, y_pred)
    print(ba, f1, p_score, r_score, mcc)
    print(classification_report(y_true, y_pred, zero_division=np.nan))
    return ba, f1, p_score, r_score, mcc

In [6]:
train_path      = '/home/hnguyen/Documents/PhD/Part3/Data/numpy_data/01_PB2/'
test_path       = '/home/hnguyen/Documents/PhD/Part3/Data/numpy_data/01_PB2/' 

In [7]:
X_train = np.load(train_path + 'X_train_onehot.npy')
y_train = np.load(train_path + 'y_train.npy')

In [8]:
X_test_high_quality = np.load(test_path + 'X_test_onehot.npy')
y_test_high_quality = np.load(test_path + 'y_test.npy')

In [9]:
print("Train data shape:", X_train.shape, y_train.shape)
print("Test High-quality Data Shape:", X_test_high_quality.shape, y_test_high_quality.shape)

Train data shape: (78732, 5, 2400) (78732,)
Test High-quality Data Shape: (83325, 5, 2400) (83325,)


In [10]:

human_index = np.where(y_train == 0, True, False)
avian_index = np.where(y_train == 1, True, False)
mammal_index = np.where(y_train == 2, True, False)

X_CV_train_human = X_train[human_index]
X_CV_train_avian = X_train[avian_index]
X_CV_train_mammal = X_train[mammal_index]

background_human = X_CV_train_human[np.random.choice(X_CV_train_human.shape[0], 350, replace=False)]
background_avian = X_CV_train_avian[np.random.choice(X_CV_train_avian.shape[0], 350, replace=False)]
background_mammal = X_CV_train_mammal[np.random.choice(X_CV_train_mammal.shape[0], 300, replace=False)]

background_data = np.concatenate((background_human, background_avian, background_mammal), axis=0)

In [11]:
n_out = len(np.unique(y_train))
seq_len = X_train.shape[2]
res_len = X_train.shape[1]
patch_size = (12, res_len)
epochs = 35
batch_size = 256
emb_dim = 64
final_hidden_size = 24
n_splits = 10

In [12]:
cv_cols = ["Model", "Balanced Accuracy", "F1-Score (Macro)", "Precision (Macro)", "Recall (Macro)", "MCC"]
param_results_high_quality = []

In [13]:
splitter = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=1, random_state=0)

In [14]:
seed = 0
set_seed(seed) # old is 0

Set Global Seed



In [15]:
wsn_clf = WaveSeekerClassifier(
    n_channels=1,
    seq_L=seq_len,
    res_L=res_len,
    patch_size=patch_size,
    n_out=n_out,
    batch_size=batch_size,
    emb_dim=emb_dim,
    final_hidden_size=final_hidden_size,
    epochs=epochs,
    patch_mode="patch",
    wavelet_names=["sym4"],
    n_blocks=1,
    lr=0.0025,)

In [16]:
# clf_load_weight.summary()

In [17]:
wsn_clf.load_weights("/home/hnguyen/Documents/PhD/Part3/Data/model_weights/01_PB2/Ablation_weight_0_Baseline.pt")

INFO | WaveSeekerNet.model | Initializing model...
INFO | WaveSeekerNet.model | Using device: cuda:0


In [18]:
_, logits = wsn_clf.predict(X_test_high_quality, return_logits=True)

In [19]:
ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, np.argmax(logits, axis=1))

0.954472453842666 0.92419477991587 0.8982995914642359 0.954472453842666 0.9421877935572929
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     73749
           1       0.96      1.00      0.98      7495
           2       0.73      0.88      0.80      2081

    accuracy                           0.99     83325
   macro avg       0.90      0.95      0.92     83325
weighted avg       0.99      0.99      0.99     83325



In [20]:
check = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/data/section1/01_PB2/post_2020_logits_fold_0.npy')
ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, np.argmax(check, axis=1))

0.954472453842666 0.92419477991587 0.8982995914642359 0.954472453842666 0.9421877935572929
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     73749
           1       0.96      1.00      0.98      7495
           2       0.73      0.88      0.80      2081

    accuracy                           0.99     83325
   macro avg       0.90      0.95      0.92     83325
weighted avg       0.99      0.99      0.99     83325



In [21]:
np.testing.assert_allclose(logits, check, rtol=1e-3, atol=1e-2)

In [22]:
logits[1000]

array([ 2.1576748 , -1.3759941 , -0.03716573], dtype=float32)

In [23]:
check[1000]

array([ 2.1576655 , -1.375987  , -0.03716547], dtype=float32)

In [24]:
# model_explain = wsn_clf.explain(X_test_high_quality[:100], background_data)

In [25]:
# model_explain.shape

In [27]:
for kfold_index, (train, test) in enumerate(splitter.split(X_train, y_train)):
    X_CV_train, y_CV_train = resampling(X_train[train], y_train[train], n_downsamples=16000, n_upsamples=600) # up sampling human and avian, keep non-human mammals (set 600)

    X_CV_test  = X_train[test]
    y_CV_test  = y_train[test]
    
    print ("Train shape: ", X_CV_train.shape, y_CV_train.shape)
    print (np.transpose(np.unique(y_CV_train, return_counts=True)))
    print ("Val shape: ", X_CV_test.shape, y_CV_test.shape)
    print (np.transpose(np.unique(y_CV_test, return_counts=True)))

    
    clf = WaveSeekerClassifier(
        n_channels=1,
        seq_L=seq_len,
        res_L=res_len,
        patch_size=patch_size,
        n_out=n_out,
        batch_size=batch_size,
        emb_dim=emb_dim,
        final_hidden_size=final_hidden_size,
        epochs=epochs,
        patch_mode="patch",
        wavelet_names=["sym4"],
        n_blocks=1,
        lr=0.0025)
    
    model_name = "Baseline"
    #clf.summary()

    clf.fit(X_CV_train, y_CV_train, X_CV_test, y_CV_test)#, save_path=model_weight)
    print("%s Result:" % model_name)
    print("High Quality (Post 2020)")
    ctest_high_quality = clf.predict(X_test_high_quality)
    
    ba_main, f1_main, p_score_main, r_score_main, mcc_main = get_score(y_test_high_quality, ctest_high_quality)
    param_results_high_quality.append((model_name, ba_main, f1_main, p_score_main, r_score_main, mcc_main))
    break 

INFO | WaveSeekerNet.utils | Data Shape Before Sampling: (70858, 5, 2400) (70858,)
INFO | WaveSeekerNet.utils | Subtype/Host: 0, count: 42381, downsampling: (16000, 5, 2400) (16000,)
INFO | WaveSeekerNet.utils | Subtype/Host: 1, count: 20859, downsampling: (16000, 5, 2400) (16000,)
INFO | WaveSeekerNet.utils | Subtype/Host: 2, count: 7618, keep: (7618, 5, 2400) (7618,)
Train shape:  (39618, 5, 2400) (39618,)
[[    0 16000]
 [    1 16000]
 [    2  7618]]
Val shape:  (7874, 5, 2400) (7874,)
[[   0 4709]
 [   1 2318]
 [   2  847]]
INFO | WaveSeekerNet.model | Using device: cuda:0
INFO | WaveSeekerNet.model | Trainable parameters: 1179123 / 1179123 total
INFO | WaveSeekerNet.model | Epoch 1/35 | BCE: 1.0588 | KAN: 0.0789 | SMoE: 0.1050 | Val Loss: 1.0209 | Val BA: 0.3333 | Train: 74.3s | Infer: 2.8s
INFO | WaveSeekerNet.model | Epoch 2/35 | BCE: 1.0268 | KAN: 0.0549 | SMoE: 0.0417 | Val Loss: 0.8983 | Val BA: 0.5130 | Train: 73.3s | Infer: 2.8s
INFO | WaveSeekerNet.model | Epoch 3/35 | BCE

In [61]:
test = np.load('/home/hnguyen/Documents/PhD/Part1/WaveSeekerNet/Data_Preparation/Protein/06_NA_Protein_Subtype/encoded_arr/X_test_high_onehot.npy')

In [62]:
test.shape

(28219, 21, 495)

In [63]:
X_test_protein, _ = protein_fasta_to_one_hot(
    fasta_path="/home/hnguyen/Documents/PhD/Part1/WaveSeekerNet/Data_Preparation/Protein/06_NA_Protein_Subtype/for_blastp_get_fold_training_data/6_NA_High_Quality_Test_Sequence.fasta",
    seq_len=495,
    chunk_size=10000,
    out_filename="X_train_protein.npy"
)

INFO | WaveSeekerNet.utils | Found 28219 protein sequences in FASTA file: /home/hnguyen/Documents/PhD/Part1/WaveSeekerNet/Data_Preparation/Protein/06_NA_Protein_Subtype/for_blastp_get_fold_training_data/6_NA_High_Quality_Test_Sequence.fasta
INFO | WaveSeekerNet.utils | Initializing disk-backed memory-mapped array at X_train_protein.npy...
INFO | WaveSeekerNet.utils | Encoded protein records 0 to 9999...
INFO | WaveSeekerNet.utils | Encoded protein records 10000 to 19999...
INFO | WaveSeekerNet.utils | Encoded final protein records 20000 to 28218.
INFO | WaveSeekerNet.utils | Protein encoding complete. Dataset saved to X_train_protein.npy


In [64]:
X_test_protein.shape

(28219, 21, 495)

In [65]:
np.testing.assert_array_equal(X_test_protein, test)